In [33]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

from starter_pack.code.align_images  import *
from starter_pack.code.utils  import *
# print(align_images)
# print(type(align_images))

In [34]:
def gaussian_1d(sigma):
    r = int(np.ceil(3 * sigma))
    x = np.arange(-r, r + 1)
    g = np.exp(-x ** 2 / (2 * sigma ** 2))
    return g / g.sum()

In [35]:
def conv1d_reflect(img, k, axis):
    """Convolve a 2D image with 1D kernel k along `axis`, reflect padding, same size."""
    r = len(k) // 2
    pad = [(0, 0), (0, 0)]
    pad[axis] = (r, r)
    p = np.pad(img, pad, mode="reflect")
    k = k[::-1]                      # convolution = correlation with flipped kernel
    out = np.zeros_like(img, dtype=np.float64)
    n = img.shape[axis]
    for i, w in enumerate(k):        # loop over kernel taps only, whole-image ops
        sl = [slice(None), slice(None)]
        sl[axis] = slice(i, i + n)
        out += w * p[tuple(sl)]
    return out


In [36]:
def blur(img, sigma):
    """Separable Gaussian blur on a gray (H,W) or color (H,W,C) image, per channel."""
    k = gaussian_1d(sigma)
    if img.ndim == 2:
        return conv1d_reflect(conv1d_reflect(img, k, 0), k, 1)
    return np.stack([blur(img[..., c], sigma) for c in range(img.shape[2])], axis=2)

In [37]:
def cutoff_cpi(sigma, width):
    """Frequency (cycles/image) where a Gaussian's gain = 1/2 (the paper's definition)."""
    return np.sqrt(np.log(2) / 2) / (np.pi * sigma) * width
 
 
def make_hybrid(im_far, im_near, sigma_low, sigma_high):
    low = blur(im_far, sigma_low)                 # seen from far away
    high = im_near - blur(im_near, sigma_high)    # seen up close
    hybrid = np.clip(low + high, 0, 1)
    return low, high, hybrid

In [38]:
def log_fft(img):
    gray = img if img.ndim == 2 else cv2.cvtColor(img.astype(np.float32), cv2.COLOR_RGB2GRAY)
    return np.log(np.abs(np.fft.fftshift(np.fft.fft2(gray))) + 1e-8)

In [39]:
def fit_size(im, max_side=1000):
    """Bring an image to <= max_side on its longer side. Pick align points AFTER this."""
    s = max_side / max(im.shape[:2])
    if s >= 1:
        return im
    return cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)

In [40]:
# def match_height(im, target_h):
#     s = target_h / im.shape[0]
#     return cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)


In [41]:
def run(name, path_far, path_near, sigma_low, sigma_high,
        pts_far=None, pts_near=None, gray=True):
    """pts_far / pts_near: two (x, y) points in each image (e.g. the two eyes).
    If None, you click them (pick_points; local Python only)."""
    im1 = fit_size(load_image(path_far, gray=gray))
    im2 = fit_size(load_image(path_near, gray=gray))
    if pts_far is None:
        pts_far = pick_points(im1)
    if pts_near is None:
        pts_near = pick_points(im2)
    print(f"[{name}] pts_far={pts_far}  pts_near={pts_near}")   # copy these into the call to reuse
 
    im2_al = align_images(im1, im2, pts_far, pts_near)          # im2 warped onto im1
 
    low, high, hybrid = make_hybrid(im1, im2_al, sigma_low, sigma_high)
 
    W = hybrid.shape[1]
    print(f"[{name}] sigma_low={sigma_low} (fc={cutoff_cpi(sigma_low, W):.1f} c/i), "
          f"sigma_high={sigma_high} (fc={cutoff_cpi(sigma_high, W):.1f} c/i)")
 
    # ---- process figure: images on top, log-magnitude FFT below ----
    imgs = [im1, im2_al, low, high + 0.5, hybrid]               # +0.5: zero -> mid-gray
    titles = ["Image 1 (low-pass source)", "Image 2, aligned", f"Low-pass σ={sigma_low}",
              f"High-pass σ={sigma_high}", "Hybrid"]
    ffts = [log_fft(im1), log_fft(im2_al), log_fft(low), log_fft(high), log_fft(hybrid)]
 
    fig, ax = plt.subplots(2, 5, figsize=(20, 8))
    for i in range(5):
        shown = np.clip(imgs[i], 0, 1)
        ax[0, i].imshow(shown, cmap="gray" if shown.ndim == 2 else None, vmin=0, vmax=1)
        ax[0, i].set_title(titles[i])
        ax[0, i].axis("off")
        ax[1, i].imshow(ffts[i], cmap="gray")
        ax[1, i].set_title("log |FFT|")
        ax[1, i].axis("off")
    plt.tight_layout()
    plt.savefig(f"{name}_process.png", dpi=120)
    plt.show()
 
    save_image(f"{name}_hybrid.png", hybrid)
    small = cv2.resize(hybrid.astype(np.float32), None, fx=0.2, fy=0.2, interpolation=cv2.INTER_AREA)
    save_image(f"{name}_hybrid_small.png", small)               # simulates viewing from far away
    return hybrid

In [46]:
run(
    "man_cat",
    "starter_pack/images/part2_frequencies/hybrid_man.jpg",
    "starter_pack/images/part2_frequencies/hybrid_cat.jpg",
    sigma_low=2,
    sigma_high=2
)

[man_cat] pts_far=[(394.3376623376622, 332.2922077922077), (327.7792207792206, 340.4090909090909)]  pts_near=[(499.5, 268.04838709677426), (443.04838709677415, 224.5000000000001)]
[man_cat] sigma_low=2 (fc=67.0 c/i), sigma_high=2 (fc=67.0 c/i)


array([[0.30822926, 0.32723385, 0.35626149, ..., 0.41916754, 0.41502567,
        0.41065547],
       [0.37543407, 0.35188535, 0.35505799, ..., 0.41420992, 0.41134065,
        0.4083693 ],
       [0.38938956, 0.37531658, 0.35686357, ..., 0.41183812, 0.41162031,
        0.41190406],
       ...,
       [0.00125309, 0.00862625, 0.00699234, ..., 0.13818873, 0.14277482,
        0.13667749],
       [0.01096002, 0.01769863, 0.01143353, ..., 0.13860718, 0.14096919,
        0.14128821],
       [0.01752046, 0.03239126, 0.02755104, ..., 0.13985341, 0.14242942,
        0.14227371]], shape=(1000, 715))

In [68]:
run(
    "marlin_albert",
    
    "albert.jpg",
    "marlin.jpg",
    sigma_low=2,
    sigma_high=2
)

[marlin_albert] pts_far=[(231.61201298701303, 210.78571428571428), (192.39610389610397, 208.38474025974028)]  pts_near=[(243.3709677419355, 156.72903225806448), (203.6548387096774, 160.21290322580643)]
[marlin_albert] sigma_low=2 (fc=37.9 c/i), sigma_high=2 (fc=37.9 c/i)


array([[0.16912048, 0.2026663 , 0.21981662, ..., 0.15422048, 0.17090713,
        0.20031341],
       [0.14946419, 0.17251684, 0.18661441, ..., 0.19091083, 0.18323743,
        0.18863879],
       [0.17716406, 0.16634136, 0.15374753, ..., 0.20542249, 0.2062573 ,
        0.24558864],
       ...,
       [0.11859318, 0.18995389, 0.24822082, ..., 0.14949183, 0.17243653,
        0.18852956],
       [0.14879664, 0.31612389, 0.45160752, ..., 0.18667815, 0.19678709,
        0.203943  ],
       [0.30397648, 0.50647745, 0.60137391, ..., 0.22101811, 0.21473546,
        0.19535081]], shape=(493, 405))

In [65]:
run(
    "cat_panda",
    "cat.png",
    "panda.png",
    
    sigma_low=1,
    sigma_high=6
)

[cat_panda] pts_far=[(83.65584415584416, 58.72077922077921), (55.188311688311686, 61.62987012987011)]  pts_near=[(63.91558441558442, 63.08441558441557), (48.538961038961034, 62.04545454545453)]
[cat_panda] sigma_low=1 (fc=24.0 c/i), sigma_high=6 (fc=4.0 c/i)


array([[0.34305339, 0.37801519, 0.42454143, ..., 0.54314405, 0.37700271,
        0.27189147],
       [0.33545884, 0.37577623, 0.42825979, ..., 0.53838165, 0.35651649,
        0.2501386 ],
       [0.31091378, 0.35618512, 0.42578021, ..., 0.50836528, 0.31945529,
        0.22209052],
       ...,
       [0.24359605, 0.27298826, 0.33347316, ..., 0.68140589, 0.6969162 ,
        0.71207296],
       [0.22760842, 0.25648424, 0.30924321, ..., 0.70730856, 0.71701104,
        0.71332631],
       [0.22873691, 0.24973351, 0.29257923, ..., 0.72504026, 0.73502695,
        0.72033149]], shape=(128, 128))

In [43]:
%matplotlib qt